# ONG v3 - Pilot: BioCLIP-2 ViT-L/14 (recipe bersih, prior-respecting)

Re-run **fair** untuk BioCLIP-2 dengan recipe yang sama seperti pilot DINOv2 -
`--sampler-power 0 --loss ce` (tanpa double-balancing; menghormati prior Bulbophyllum/
Dendrobium). BioCLIP-2 adalah backbone **biology-pretrained** dan top pick untuk RETRIEVAL di
bake-off lama (species R@5 87.5%, genus R@5 98.3%). Pilot ini mengukur apakah genus
classifier-nya juga membaik dengan recipe bersih + checkpoint global-selected.

Menyimpan **`best_model_macro.pth` + `best_model_global.pth`**, meng-eval **keduanya** pada
test split frozen. BioCLIP-2 dievaluasi lewat cabang **`--arch openclip`** di 13_evaluate.py
(beda dgn backbone timm) - sudah otomatis di Sel 4.

### Auto-resume (anti-crash Colab)
Notebook ini jalan dengan **`--resume`**: tiap epoch, state lengkap (bobot model + EMA + posisi
LR-schedule + metrik terbaik + history) disimpan ke Drive di
`models/bioclip2/ckpt_resume.pth`. Kalau sesi **putus/crash** (mis. ketiduran), cukup buka lagi
notebook -> **Run all**: training **lanjut dari epoch terakhir yang selesai**, bukan dari nol.
Saat training tuntas, checkpoint resume otomatis dihapus. (Catatan: optimizer momentum & AMP
scaler di-*reset* saat resume - efeknya sangat kecil untuk fine-tuning; bobot/EMA/jadwal-LR tetap
lanjut persis. Sediakan ~3 GB ruang Drive untuk file resume.)

**Pembanding (test split frozen yang SAMA, ~57-58 genus):**

| Model | global T1 | macro T1 | species R@5 | genus R@5 |
|---|---|---|---|---|
| effb4 baseline (OPTIMISTIC, ada overlap) | 90.1 | **74.0** | 73.4 | 94.1 |
| DINOv2-L (pilot, recipe bersih) | 88.9 | 66.9 | 86.6 | **98.7** |
| BioCLIP-2 (bake-off lama, overfit) | 54.8 | 57.2 | **87.5** | 98.3 |

**Sebelum run**, upload ke Drive `MyDrive/ONG_v3/`:
`scripts/03_train_bakeoff_colab.py`, `scripts/13_evaluate.py`,
`data/splits/{train,val,test}_live.csv`, `data/splits/all_images.csv`, `photos.zip`.
Pakai L4 (Runtime -> Change runtime type -> L4), lalu **Run all**. BioCLIP-2 @224 -> relatif
ringan (kira-kira 10-15 compute unit).

In [1]:
# Sel 1 - Mount Drive, install deps, cek GPU
from google.colab import drive; drive.mount('/content/drive')
!pip -q install -U timm open_clip_torch faiss-cpu
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > L4, lalu Run all lagi.'

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
GPU: NVIDIA L4


In [2]:
# Sel 2 - Set ROOT + parameter backbone, verifikasi file di Drive
import os
ROOT = '/content/drive/MyDrive/ONG_v3'

# Parameter HARUS sama dgn REGISTRY di 03_train_bakeoff_colab.py (train==eval konsisten)
MODEL     = 'bioclip2'
EVAL_NAME = 'hf-hub:imageomics/bioclip-2'
IMG       = 224
ARCH      = 'openclip'   # bioclip2 = openclip; backbone timm = timm
print(f'Backbone: {MODEL}  ({EVAL_NAME})  img={IMG}  arch={ARCH}')

need = ['scripts/03_train_bakeoff_colab.py', 'scripts/13_evaluate.py',
        'data/splits/train_live.csv', 'data/splits/val_live.csv',
        'data/splits/test_live.csv', 'data/splits/all_images.csv', 'photos.zip']
status = {f: os.path.exists(f'{ROOT}/{f}') for f in need}
print('present:', status)
missing = [f for f, ok in status.items() if not ok]
assert not missing, f'Missing di {ROOT}/: {missing} - upload dulu, lalu Run all lagi.'
print('Semua file siap.')

Backbone: bioclip2  (hf-hub:imageomics/bioclip-2)  img=224  arch=openclip
present: {'scripts/03_train_bakeoff_colab.py': True, 'scripts/13_evaluate.py': True, 'data/splits/train_live.csv': True, 'data/splits/val_live.csv': True, 'data/splits/test_live.csv': True, 'data/splits/all_images.csv': True, 'photos.zip': True}
Semua file siap.


In [5]:
# Sel 3 - Unzip photos ke disk lokal cepat (/content/photos/{Genus}/*.jpg)
!unzip -q -o "{ROOT}/photos.zip" -d /content/
nested = '/content/photos/photos'
if os.path.isdir(nested):
    import shutil
    for it in os.listdir(nested):
        shutil.move(os.path.join(nested, it), '/content/photos')
    os.rmdir(nested)
assert os.path.isdir('/content/photos'), 'Tidak ada /content/photos setelah unzip - cek layout zip.'
print('folder genus:', len(os.listdir('/content/photos')))

folder genus: 127


In [ ]:
# Sel 4 - Train (recipe bersih) + eval KEDUA checkpoint pada test split frozen
# --img-size diteruskan ke train DAN eval supaya tidak bisa drift.
# --resume: checkpoint tiap epoch ke Drive (models/{MODEL}/ckpt_resume.pth); kalau crash, Run all lagi -> lanjut dari epoch terakhir, bukan dari nol.
!python {ROOT}/scripts/03_train_bakeoff_colab.py --model {MODEL} --drive-root {ROOT} \
    --photos-root /content/photos --img-size {IMG} --sampler-power 0 --loss ce --resume

# bioclip2 dievaluasi lewat cabang --arch openclip; backbone timm tidak butuh flag itu.
arch_flag = f'--arch {ARCH} ' if ARCH == 'openclip' else ''
for tag in ['macro', 'global']:
    ckpt = f'{ROOT}/models/{MODEL}/best_model_{tag}.pth'
    out  = f'{ROOT}/eval/{MODEL}' if tag == 'macro' else f'{ROOT}/eval/{MODEL}_{tag}'
    assert os.path.exists(ckpt), f'Checkpoint hilang: {ckpt} - training gagal (lihat log di atas).'
    print(f'\n{"="*70}\n[EVAL] {MODEL} [{tag}-selected]\n{"="*70}')
    get_ipython().system(
        f'python {ROOT}/scripts/13_evaluate.py {arch_flag}--model {EVAL_NAME} --img-size {IMG} '
        f'--checkpoint {ckpt} --vocab {ROOT}/models/{MODEL}/vocab.json '
        f'--splits-dir {ROOT}/data/splits --photos-root /content/photos --out-dir {out}'
    )

Device: cuda | Model: bioclip2 (hf-hub:imageomics/bioclip-2) | img=224 | bs=32
GPU: NVIDIA L4
Genera: 120 | Train: 11,677 | Val: 2,629
open_clip_config.json: 100% 534/534 [00:00<00:00, 2.20MB/s]
open_clip_model.safetensors: 100% 1.71G/1.71G [00:04<00:00, 374MB/s]
Loss: CrossEntropy (label_smoothing=0.1)
RESUME on, but no ckpt_resume.pth yet — starting fresh; will checkpoint each epoch.

Phase warmup | epochs=3 | lr_head=0.001 lr_bb=0.0 | trainable=92,280
  [warmup 1/3] loss=1.9613 val_macro=29.25% (ema 1.01%) val_top1=68.35% (ema 0.49%) top5=86.12% 264s
    >> best val_macro=29.25% (raw) — saved
    >> best val_top1=68.35% (raw) — saved
  [warmup 2/3] loss=1.5744 val_macro=34.12% (ema 2.86%) val_top1=70.94% (ema 5.55%) top5=86.57% 269s
    >> best val_macro=34.12% (raw) — saved
    >> best val_top1=70.94% (raw) — saved
  [warmup 3/3] loss=1.4742 val_macro=34.25% (ema 6.33%) val_top1=72.38% (ema 21.68%) top5=87.68% 266s
    >> best val_macro=34.25% (raw) — saved
    >> best val_top1=72.

In [ ]:
# Sel 5 - Tabel banding (baca setiap eval/<dir>/results.json) + simpan ringkasan markdown
import json, datetime
from pathlib import Path

BASE = dict(name='baseline effb4 (FROZEN)', macro_top1=74.0, macro_top5=84.8,
            global_top1=90.1, sp_r5=73.4, gn_r5=94.1, ece=0.082)
PRETTY = {'bioclip2': 'BioCLIP-2 ViT-L/14', 'dinov2l': 'DINOv2 ViT-L/14',
          'convnextv2l': 'ConvNeXt-V2-L', 'effnetv2l': 'EfficientNetV2-L'}

def pretty(name):
    suf = ''
    if name.endswith('_global'):
        name, suf = name[:-7], ' [global-sel]'
    return PRETTY.get(name, name) + suf

def read(p):
    r = json.loads(Path(p).read_text())
    c = r.get('classification', {}) or {}
    q = r.get('retrieval', {}) or {}
    pc = lambda d, k: round(d[k] * 100, 1) if d.get(k) is not None else None
    return dict(macro_top1=pc(c, 'macro_top1'), macro_top5=pc(c, 'macro_top5'),
                global_top1=pc(c, 'global_top1'), sp_r5=pc(q, 'species_recall@5'),
                gn_r5=pc(q, 'genus_recall@5'),
                ece=(round(c['ece'], 3) if c.get('ece') is not None else None))

evdir = Path(ROOT) / 'eval'
found = sorted(p.parent.name for p in evdir.glob('*/results.json')
               if not p.parent.name.startswith('_'))
f2 = lambda x: '-' if x is None else f'{x}'
date = datetime.date.today().isoformat()
L = [f'## {date} - Pilot {MODEL} (recipe bersih; protokol split frozen)',
     '',
     'Baseline = effb4 FROZEN (OPTIMISTIC: train/test overlap; macro>=5 76.2%). '
     '`[global-sel]` = checkpoint dipilih by val global top-1.',
     '',
     '| Model | macro T1 | macro T5 | global T1 | species R@5 | genus R@5 | ECE |',
     '|-------|----------|----------|-----------|-------------|-----------|-----|',
     f"| {BASE['name']} | **{BASE['macro_top1']}** | {BASE['macro_top5']} | {BASE['global_top1']} | {BASE['sp_r5']} | {BASE['gn_r5']} | {BASE['ece']} |"]
best_cls = best_ret = None
for name in found:
    r = read(evdir / name / 'results.json')
    L.append(f"| {pretty(name)} | **{f2(r['macro_top1'])}** | {f2(r['macro_top5'])} | "
             f"{f2(r['global_top1'])} | {f2(r['sp_r5'])} | {f2(r['gn_r5'])} | {f2(r['ece'])} |")
    if r['macro_top1'] is not None and (best_cls is None or r['macro_top1'] > best_cls[1]):
        best_cls = (name, r['macro_top1'])
    if r['sp_r5'] is not None and (best_ret is None or r['sp_r5'] > best_ret[1]):
        best_ret = (name, r['sp_r5'])
L.append('')
if best_cls:
    d = round(best_cls[1] - BASE['macro_top1'], 1)
    L.append(f"- **Best genus (macro top-1):** {pretty(best_cls[0])} = {best_cls[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
if best_ret:
    d = round(best_ret[1] - BASE['sp_r5'], 1)
    L.append(f"- **Best retrieval (species R@5):** {pretty(best_ret[0])} = {best_ret[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
snippet = '\n'.join(L)
out = evdir / f'pilot_{MODEL}_summary_{date}.md'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(snippet, encoding='utf-8')
print(snippet)
print(f'\nSaved -> {out}  (download via Files panel; tempel ke ONGv3_progress.Rmd)')